# P52 Asset Generation

Generates multiple cat diffusion sequences and multiple robot pen uncapping episodes for p52 grid.

## Setup

In [ ]:
import subprocess, os, sys

UV = '/root/.local/bin/uv'

# Clone openpi if not already present
if not os.path.exists('/root/openpi'):
    subprocess.run([
        'git', 'clone', '--recurse-submodules',
        'https://github.com/Physical-Intelligence/openpi.git', '/root/openpi'
    ], check=True)
    print("Cloned openpi")
else:
    print("openpi already cloned")

subprocess.run([
    UV, 'pip', 'install', '--system',
    '-e', '/root/openpi[dev]',
], check=True)


In [ ]:
subprocess.run([
    UV, 'pip', 'install', '--system',
    'matplotlib', 'datasets', 'diffusers', 'transformers',
    'accelerate', 'safetensors', 'sentencepiece',
    'typing_extensions>=4.12',
], check=True)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from datasets import load_dataset
from PIL import Image
import copy, math, time, os
from tqdm import tqdm

import torch
import torch.nn.functional as F

from openpi.training import config as _config
from openpi.policies import policy_config as _policy_config
from openpi.policies import aloha_policy
from openpi.shared import download
from openpi import transforms as _transforms
from openpi.shared import normalize as _normalize


In [ ]:
def frame_to_obs(frame):
    def to_chw(img):
        arr = np.array(img)
        return arr.transpose(2, 0, 1)
    return {
        "images": {
            "cam_high": to_chw(frame["observation.images.cam_high"]),
            "cam_left_wrist": to_chw(frame["observation.images.cam_left_wrist"]),
            "cam_right_wrist": to_chw(frame["observation.images.cam_right_wrist"]),
        },
        "state": np.array(frame["observation.state"], dtype=np.float32),
        "prompt": "uncap the pen",
    }


## 1. Generate Multiple Robot Pen Uncapping Episodes

Save cam_low frames for each episode, plus run the denoising loop to get heatmap trajectories.

In [ ]:
BASE_DIR = '/workspace/videos/_2026/vla/p52_assets'
os.makedirs(BASE_DIR, exist_ok=True)

ds = load_dataset("physical-intelligence/aloha_pen_uncap_diverse", split="train")
episode_indices = sorted(set(ds["episode_index"]))
print(f"Total rows: {len(ds)}")
print(f"Available episodes: {episode_indices}")


In [ ]:
# Load model
config = _config.get_config("pi0_aloha_pen_uncap")
CHECKPOINT_URI = "gs://openpi-assets/checkpoints/pi0_base"
checkpoint_dir = str(download.maybe_download(CHECKPOINT_URI))
policy = _policy_config.create_trained_policy(config, checkpoint_dir)
print("Policy loaded!")


In [ ]:
# How many episodes to render
NUM_EPISODES = 16
EVAL_FRAME_IDX = 150

ep_indices_arr = np.array(ds["episode_index"])

for ep_idx in episode_indices[:NUM_EPISODES]:
    print(f"\n=== Episode {ep_idx} ===")
    
    # Select episode frames
    mask = ep_indices_arr == ep_idx
    row_indices = np.where(mask)[0]
    episode = ds.select(row_indices)
    print(f"  {len(episode)} frames")
    
    # --- Save cam_low frames ---
    cam_dir = f'{BASE_DIR}/robot_ep{ep_idx}'
    os.makedirs(cam_dir, exist_ok=True)
    
    for idx in tqdm(range(len(episode)), desc=f"  cam_low ep{ep_idx}"):
        frame_i = episode[idx]
        img = frame_i["observation.images.cam_low"]
        img.save(f'{cam_dir}/{idx:03d}.png')
    
    # --- Run denoising to get heatmap trajectory ---
    heatmap_dir = f'{BASE_DIR}/heatmap_ep{ep_idx}'
    os.makedirs(heatmap_dir, exist_ok=True)
    
    eval_frame = min(EVAL_FRAME_IDX, len(episode) - 1)
    frame = episode[eval_frame]
    
    # Use different noise seed per episode for variety
    rng = np.random.RandomState(42 + ep_idx)
    noise = rng.randn(50, 32).astype(np.float32)
    
    result = policy.infer(frame_to_obs(frame), noise=noise)
    
    # Build interpolated trajectory (noise -> final actions)
    target_actions = result["actions"].astype(np.float32)
    x0 = noise.copy()
    num_steps = 10
    
    trajectory = [x0.copy()]
    for step in range(num_steps):
        alpha = (step + 1) / num_steps
        x_next = (1.0 - alpha) * x0 + alpha * np.zeros_like(x0)
        x_next[:, :target_actions.shape[-1]] = (1.0 - alpha) * x0[:, :target_actions.shape[-1]] + alpha * target_actions
        trajectory.append(x_next.copy())
    
    # Save heatmaps
    for i, traj in enumerate(trajectory):
        matplotlib.image.imsave(f'{heatmap_dir}/{i:02d}.png', traj[:, :14].T, cmap="viridis")
    
    # Save full trajectory array
    np.save(f'{heatmap_dir}/trajectory.npy', np.array(trajectory))
    
    print(f"  Saved {len(episode)} cam_low frames + {len(trajectory)} heatmaps")

print(f"\nDone! {NUM_EPISODES} episodes saved to {BASE_DIR}/")


## 2. Generate Multiple Cat Diffusion Sequences\n\nUses Realistic Vision V5.1 (Stable Diffusion) with different seeds to produce diverse cat images.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from PIL import Image
from pathlib import Path

PROMPT = "A happy gray tabby cat looking at the camera, photorealistic, 8k"
NEGATIVE_PROMPT = "cartoon, illustration, painting, drawing, anime, low quality"
NUM_STEPS = 100
GUIDANCE_SCALE = 7.0
NUM_CAT_SEQUENCES = 16

pipe = StableDiffusionPipeline.from_pretrained(
    "SG161222/Realistic_Vision_V5.1_noVAE",
    torch_dtype=torch.float16,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type="dpmsolver++",
)
pipe.to("cuda")
print("Stable Diffusion pipeline loaded!")


In [ ]:
SEEDS = [5, 12, 42, 77, 101, 256, 314, 628, 999, 1234, 2048, 3141, 4096, 5555, 7777, 9999]

for seq_idx in range(NUM_CAT_SEQUENCES):
    seed = SEEDS[seq_idx]
    output_dir = Path(f'{BASE_DIR}/cat_seq{seq_idx}')
    output_dir.mkdir(exist_ok=True)
    
    print(f"\n=== Cat sequence {seq_idx} (seed={seed}) ===")
    
    intermediate_latents = []
    
    def capture_latents(pipe, step_index, timestep, callback_kwargs):
        intermediate_latents.append(callback_kwargs["latents"].detach().clone())
        return callback_kwargs
    
    generator = torch.Generator(device="cuda").manual_seed(seed)
    
    result = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
        callback_on_step_end=capture_latents,
    )
    
    final_image = result.images[0]
    final_image.save(output_dir / "final.png")
    
    # Decode intermediates
    step_images = []
    for i, latents in enumerate(intermediate_latents):
        with torch.no_grad():
            decoded = pipe.vae.decode(latents / pipe.vae.config.scaling_factor, return_dict=False)[0]
            decoded = (decoded / 2 + 0.5).clamp(0, 1)
            img = decoded[0].permute(1, 2, 0).cpu().float().numpy()
            img = Image.fromarray((img * 255).astype("uint8"))
        step_images.append(img)
        img.save(output_dir / f"step_{i:03d}.png")
    
    print(f"  Saved {len(step_images)} steps + final to {output_dir}/")

print(f"\nDone! {NUM_CAT_SEQUENCES} cat sequences saved.")


## 3. Summary\n\nAssets saved to `{BASE_DIR}/`:\n- `robot_ep{N}/` — cam_low frames (000.png, 001.png, ...) for each episode\n- `heatmap_ep{N}/` — denoising heatmaps (00.png ... 10.png) + trajectory.npy\n- `cat_seq{N}/` — diffusion steps (step_000.png ... step_099.png) + final.png